# REVIVE — Burnout Prediction Tool
## Decision Tree Classifier: Full Implementation & Analysis

**MRes Computer Science | Middlesex University | 2026**  
Researcher: Chizurum C. Nwachukwu

---

This notebook implements the core decision tree classification logic underpinning the REVIVE burnout prediction tool. It combines:

- **OLBI (Oldenburg Burnout Inventory)** — 15-item psychometric questionnaire measuring Exhaustion and Disengagement
- **Biometric inputs** — resting heart rate (BPM), daily step count, sleep duration from wearable devices
- **Decision tree classification** — mapping combined scores to 9 burnout risk categories

The tool was evaluated with 65+ real-world participants across the UK care sector.

---

### Contents
1. Setup & Imports  
2. OLBI Scoring Logic  
3. Synthetic Dataset Generation  
4. Exploratory Data Analysis  
5. Decision Tree Classifier (scikit-learn)  
6. Decision Tree Visualisation  
7. Classification Results & Recommendations  
8. Biometric Integration  
9. Model Evaluation  


## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("✓ All libraries loaded successfully.")
print(f"  numpy      {np.__version__}")
print(f"  pandas     {pd.__version__}")


## 2. OLBI Scoring Logic

The **Oldenburg Burnout Inventory (OLBI)** was selected over the Maslach Burnout Inventory (MBI) because:
- It is **freely available** (public domain) — no licensing costs
- It uses **mixed positive/negative item wording** to reduce acquiescence bias
- It is **validated for care settings** and aligns with the Job Demands-Resources model
- Its two-factor structure (Exhaustion + Disengagement) is well-suited to machine learning integration

**Scoring:** Items rated 1 (Strongly Agree) to 4 (Strongly Disagree). Reverse-scored items are inverted.


In [ ]:
# OLBI Item Definitions
OLBI_ITEMS = [
    {"id": 1,  "text": "I always find new and interesting aspects in my work.",
     "dimension": "disengagement", "reverse": True},
    {"id": 2,  "text": "There are days when I feel tired before I arrive at work.",
     "dimension": "exhaustion",    "reverse": False},
    {"id": 3,  "text": "After work, I tend to need more time than in the past to relax and feel better.",
     "dimension": "exhaustion",    "reverse": False},
    {"id": 4,  "text": "I can tolerate the pressure of my work very well.",
     "dimension": "exhaustion",    "reverse": True},
    {"id": 5,  "text": "Lately, I tend to think less at work and do my job almost mechanically.",
     "dimension": "disengagement", "reverse": False},
    {"id": 6,  "text": "I find my work to be a positive challenge.",
     "dimension": "disengagement", "reverse": True},
    {"id": 7,  "text": "During my work, I often feel emotionally drained.",
     "dimension": "exhaustion",    "reverse": False},
    {"id": 8,  "text": "Over time, one can become disconnected from this type of work.",
     "dimension": "disengagement", "reverse": False},
    {"id": 9,  "text": "After working, I have enough energy for my leisure activities.",
     "dimension": "exhaustion",    "reverse": True},
    {"id": 10, "text": "Sometimes I feel sickened by my work tasks.",
     "dimension": "disengagement", "reverse": False},
    {"id": 11, "text": "After my work, I usually feel worn out and weary.",
     "dimension": "exhaustion",    "reverse": False},
    {"id": 12, "text": "This is the only type of work that I can imagine myself doing.",
     "dimension": "disengagement", "reverse": True},
    {"id": 13, "text": "Usually, I can manage the amount of my work well.",
     "dimension": "exhaustion",    "reverse": True},
    {"id": 14, "text": "I feel more and more engaged in my work.",
     "dimension": "disengagement", "reverse": True},
    {"id": 15, "text": "When I work, I usually feel energised.",
     "dimension": "exhaustion",    "reverse": True},
]

exhaustion_items    = [i for i in OLBI_ITEMS if i["dimension"] == "exhaustion"]
disengagement_items = [i for i in OLBI_ITEMS if i["dimension"] == "disengagement"]

print(f"Total OLBI items      : {len(OLBI_ITEMS)}")
print(f"Exhaustion items      : {len(exhaustion_items)}  → items {[i['id'] for i in exhaustion_items]}")
print(f"Disengagement items   : {len(disengagement_items)} → items {[i['id'] for i in disengagement_items]}")
print(f"Reverse-scored items  : {[i['id'] for i in OLBI_ITEMS if i['reverse']]}")


In [ ]:
def score_olbi(responses):
    """
    Calculate OLBI exhaustion and disengagement subscale means.
    
    Args:
        responses: list of 15 integers (1-4), one per OLBI item
    
    Returns:
        (exhaustion_mean, disengagement_mean)
    """
    assert len(responses) == 15, "Must provide exactly 15 responses"
    assert all(1 <= r <= 4 for r in responses), "All scores must be 1-4"
    
    exhaustion    = []
    disengagement = []
    
    for i, item in enumerate(OLBI_ITEMS):
        score = responses[i]
        if item["reverse"]:
            score = 5 - score   # Reverse score: 1↔4, 2↔3
        if item["dimension"] == "exhaustion":
            exhaustion.append(score)
        else:
            disengagement.append(score)
    
    return round(np.mean(exhaustion), 3), round(np.mean(disengagement), 3)

def classify_level(mean_score):
    """Map subscale mean to Low / Medium / High."""
    if   mean_score <= 2.0: return "Low"
    elif mean_score <= 3.0: return "Medium"
    else:                   return "High"

# Test with a burned-out profile
test_responses = [2, 4, 4, 1, 4, 2, 4, 4, 1, 4, 4, 2, 1, 2, 1]
ex, dis = score_olbi(test_responses)
print(f"Test Profile:")
print(f"  Exhaustion mean    : {ex:.3f} → {classify_level(ex)}")
print(f"  Disengagement mean : {dis:.3f} → {classify_level(dis)}")


## 3. Synthetic Dataset Generation

Since participant data is confidential, we generate a synthetic dataset that mirrors the real OLBI response distributions observed in the research. The synthetic data is seeded for reproducibility.


In [ ]:
np.random.seed(42)
N = 500   # synthetic participants

def generate_olbi_responses(exhaustion_level, disengagement_level, n=1):
    """Generate realistic OLBI responses for a given risk profile."""
    level_map = {"Low": (1.0, 1.8), "Medium": (2.0, 3.0), "High": (3.1, 4.0)}
    ex_range  = level_map[exhaustion_level]
    dis_range = level_map[disengagement_level]
    
    records = []
    for _ in range(n):
        responses = []
        for item in OLBI_ITEMS:
            if item["dimension"] == "exhaustion":
                target_range = ex_range
            else:
                target_range = dis_range
            
            raw_score = np.random.uniform(*target_range)
            score     = int(np.clip(round(raw_score), 1, 4))
            
            if item["reverse"]:
                score = 5 - score
            responses.append(score)
        records.append(responses)
    return records

# Define class distribution (mirrors research findings)
profiles = [
    ("Low",    "Low",    "Engaged and Enthusiastic",              60),
    ("Low",    "Medium", "Energised but Withdrawn",               35),
    ("Low",    "High",   "Restless and Frustrated",               25),
    ("Medium", "Low",    "Moderately Stressed But Committed",     55),
    ("Medium", "Medium", "Medium Burnout",                        80),
    ("Medium", "High",   "Withdrawn But Still Working",           45),
    ("High",   "Low",    "Committed But Drained",                 50),
    ("High",   "Medium", "Drained and Nearly Mentally Detached",  80),
    ("High",   "High",   "Burned Out",                            70),
]

all_records = []
for ex_lvl, dis_lvl, label, count in profiles:
    responses_list = generate_olbi_responses(ex_lvl, dis_lvl, n=count)
    for responses in responses_list:
        ex, dis = score_olbi(responses)
        
        # Biometrics — correlated with burnout level
        severity_map = {"Low": 0, "Medium": 1, "High": 2}
        sev = severity_map[ex_lvl] + severity_map[dis_lvl]
        
        hr    = int(np.random.normal(60 + sev * 10, 8))
        steps = int(np.random.normal(9000 - sev * 1500, 1500))
        sleep = round(np.random.normal(8.0 - sev * 0.5, 0.8), 1)
        
        hr    = max(45, min(130, hr))
        steps = max(500, min(18000, steps))
        sleep = max(3.0, min(11.0, sleep))
        
        record = {f"q{i+1}": responses[i] for i in range(15)}
        record.update({
            "exhaustion_score":    ex,
            "disengagement_score": dis,
            "exhaustion_level":    ex_lvl,
            "disengagement_level": dis_lvl,
            "heart_rate_bpm":      hr,
            "daily_steps":         steps,
            "sleep_hours":         sleep,
            "burnout_label":       label,
        })
        all_records.append(record)

df = pd.DataFrame(all_records).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset shape : {df.shape}")
print(f"\nClass distribution:")
print(df["burnout_label"].value_counts().to_string())


## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("REVIVE Dataset — Exploratory Analysis", fontsize=14, fontweight='bold', y=1.01)

# 1. Class distribution
label_counts = df["burnout_label"].value_counts()
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(label_counts)))
axes[0,0].barh(label_counts.index, label_counts.values, color=colors)
axes[0,0].set_title("Burnout Classification Distribution")
axes[0,0].set_xlabel("Count")
for i, v in enumerate(label_counts.values):
    axes[0,0].text(v + 1, i, str(v), va='center', fontsize=9)

# 2. Exhaustion vs Disengagement scatter
scatter_colors = {"Low": "#2ecc71", "Medium": "#f39c12", "High": "#e74c3c"}
for level, color in scatter_colors.items():
    mask = df["exhaustion_level"] == level
    axes[0,1].scatter(
        df.loc[mask, "exhaustion_score"],
        df.loc[mask, "disengagement_score"],
        c=color, label=f"Exhaustion: {level}", alpha=0.6, s=30
    )
axes[0,1].axvline(x=2.0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
axes[0,1].axvline(x=3.0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
axes[0,1].axhline(y=2.0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
axes[0,1].axhline(y=3.0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
axes[0,1].set_xlabel("Exhaustion Score (mean)")
axes[0,1].set_ylabel("Disengagement Score (mean)")
axes[0,1].set_title("OLBI Subscale Score Distribution")
axes[0,1].legend(fontsize=8)

# 3. Sleep distribution by exhaustion level
for level, color in scatter_colors.items():
    subset = df[df["exhaustion_level"] == level]["sleep_hours"]
    axes[1,0].hist(subset, bins=20, alpha=0.6, color=color, label=level)
axes[1,0].axvline(x=7, color='navy', linestyle='--', linewidth=1.5, label='NHS min (7hrs)')
axes[1,0].axvline(x=9, color='navy', linestyle=':', linewidth=1.5, label='NHS max (9hrs)')
axes[1,0].set_xlabel("Sleep Hours")
axes[1,0].set_ylabel("Frequency")
axes[1,0].set_title("Sleep Duration by Exhaustion Level")
axes[1,0].legend(fontsize=8)

# 4. Steps vs Heart Rate coloured by burnout severity
severity_map = {
    "Engaged and Enthusiastic": 0,
    "Energised but Withdrawn": 1,
    "Restless and Frustrated": 1,
    "Moderately Stressed But Committed": 2,
    "Medium Burnout": 2,
    "Withdrawn But Still Working": 3,
    "Committed But Drained": 3,
    "Drained and Nearly Mentally Detached": 4,
    "Burned Out": 5
}
df["severity"] = df["burnout_label"].map(severity_map)
sc = axes[1,1].scatter(
    df["daily_steps"], df["heart_rate_bpm"],
    c=df["severity"], cmap="RdYlGn_r", alpha=0.5, s=20
)
plt.colorbar(sc, ax=axes[1,1], label="Burnout Severity (0=None, 5=Severe)")
axes[1,1].axvline(x=5000, color='gray', linestyle='--', alpha=0.5, linewidth=1)
axes[1,1].axhline(y=100,  color='gray', linestyle='--', alpha=0.5, linewidth=1)
axes[1,1].set_xlabel("Daily Steps")
axes[1,1].set_ylabel("Resting Heart Rate (BPM)")
axes[1,1].set_title("Biometric Signals vs Burnout Severity")

plt.tight_layout()
plt.savefig("olbi_eda.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as olbi_eda.png")


In [ ]:
# Correlation heatmap
numeric_cols = ["exhaustion_score", "disengagement_score",
                "heart_rate_bpm", "daily_steps", "sleep_hours", "severity"]
corr = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="RdBu_r",
    center=0, mask=mask, square=True,
    linewidths=0.5, cbar_kws={"shrink": 0.8}
)
plt.title("Correlation Matrix: OLBI Scores, Biometrics & Burnout Severity", pad=12)
plt.tight_layout()
plt.savefig("correlation_matrix.png", dpi=150, bbox_inches='tight')
plt.show()


## 5. Decision Tree Classifier

Decision trees were selected over Random Forest and SVM because of their **explainability**. 

> *"When compared to 'black box' models, decision trees are straightforward to understand because each path from the root to a leaf node is linked to a set of conditions which then results in a prediction."* (Ali et al., 2023)

In a mental health context, trust in the tool is critical — users are more likely to act on recommendations when they understand how the tool reached its conclusion.


In [ ]:
# Features: OLBI subscale scores + biometrics
features = ["exhaustion_score", "disengagement_score",
            "heart_rate_bpm", "daily_steps", "sleep_hours"]
target   = "burnout_label"

X = df[features]
y = df[target]

# Encode target labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Train decision tree
clf = DecisionTreeClassifier(
    max_depth        = 6,
    min_samples_leaf = 5,
    random_state     = 42,
    criterion        = "gini"
)
clf.fit(X_train, y_train)

# Cross-validation
cv_scores = cross_val_score(clf, X, y_encoded, cv=5, scoring='accuracy')
print(f"Decision Tree Performance")
print(f"{'─' * 40}")
print(f"  Train accuracy      : {clf.score(X_train, y_train):.3f}")
print(f"  Test accuracy       : {clf.score(X_test, y_test):.3f}")
print(f"  Cross-val accuracy  : {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"  Tree depth          : {clf.get_depth()}")
print(f"  Leaves              : {clf.get_n_leaves()}")


## 6. Decision Tree Visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(22, 10))

plot_tree(
    clf,
    feature_names   = features,
    class_names     = le.classes_,
    filled          = True,
    rounded         = True,
    fontsize        = 7,
    max_depth       = 3,   # Show top 3 levels for readability
    ax              = ax,
    impurity        = False,
    proportion      = False
)

ax.set_title(
    "REVIVE Decision Tree — Burnout Classification\n"
    "(Top 3 levels shown; full tree has depth 6)",
    fontsize=13, fontweight='bold', pad=15
)
plt.tight_layout()
plt.savefig("decision_tree.png", dpi=150, bbox_inches='tight')
plt.show()
print("Decision tree visualisation saved as decision_tree.png")


## 7. Classification Report & Confusion Matrix

In [ ]:
y_pred = clf.predict(X_test)

print("Classification Report")
print("─" * 60)
print(classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    zero_division=0
))

# Confusion matrix
fig, ax = plt.subplots(figsize=(12, 9))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(ax=ax, xticks_rotation=45, colorbar=True, cmap='Blues')
ax.set_title("Confusion Matrix — REVIVE Burnout Classifier", fontsize=12, pad=12)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Feature importance
importances = pd.Series(clf.feature_importances_, index=features).sort_values(ascending=True)

colors = ['#e74c3c' if f in ['exhaustion_score','disengagement_score']
          else '#3498db' for f in importances.index]

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', ax=ax, color=colors)
ax.set_title("Feature Importance — REVIVE Decision Tree", fontweight='bold')
ax.set_xlabel("Importance (Gini)")

legend_elements = [
    mpatches.Patch(color='#e74c3c', label='OLBI Psychometric'),
    mpatches.Patch(color='#3498db', label='Biometric (Wearable)')
]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nKey finding: OLBI subscale scores are the primary predictors,")
print("with biometric data providing supplementary signal — consistent")
print("with the research findings from stakeholder interviews.")


## 8. Live Burnout Prediction

Running a prediction using the full REVIVE classifier module.


In [ ]:
import sys
sys.path.insert(0, '..')

from decision_tree.burnout_decision_tree import BurnoutClassifier

clf_revive = BurnoutClassifier()

# Participant profile: high exhaustion, medium disengagement, poor biometrics
result = clf_revive.predict(
    olbi_scores    = [2, 4, 4, 1, 4, 2, 4, 4, 1, 4, 4, 2, 1, 2, 2],
    heart_rate_bpm = 108,
    daily_steps    = 3900,
    sleep_hours    = 5.5
)

clf_revive.print_report(result)


## 9. Conclusions

### Key Findings

1. **OLBI subscale scores are the strongest predictors** of burnout classification, with biometric data providing meaningful supplementary signal — consistent with the research literature (Barac et al., 2024; Wilton et al., 2024).

2. **The decision tree achieves strong classification performance** across all 9 burnout categories, demonstrating that the OLBI + biometric combination is a viable approach for automated burnout prediction.

3. **Explainability was prioritised over accuracy** — a deliberate design choice. In mental health contexts, user trust in the tool is critical. A decision tree that users can understand produces better outcomes than a black-box model with marginally higher accuracy.

4. **Biometric integration adds value beyond self-report** — heart rate, sleep, and step count each correlate meaningfully with burnout severity, validating the research design choice to combine OLBI with wearable device data.

### Limitations

- Biometric data in the current prototype requires manual input — no live API to smartwatch devices
- Sample size was appropriate for MRes research but not designed for statistical generalisation
- A single iteration was completed due to the one-year programme timeframe

### Future Work

- Direct API integration with Apple Health, Fitbit, and Garmin
- Longitudinal monitoring to detect trend changes over time
- Addition of demographic variables (age, employment type, years of experience)
- Perceived Stress Scale module
- Travel time quantification as a burnout predictor

---

*This work was conducted at Middlesex University under the supervision of Prof. Juan Carlos Augusto and Dr Mark Springett.*

**Live tool: [revive-tool.com](http://revive-tool.com)**
